In [2]:
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import os
import re
from datasets import load_dataset, DatasetDict, Dataset 

from vllm import LLM, SamplingParams

from tqdm import tqdm

from llm_fairness.utils import load_resume_data, load_resume_data_with_race, load_llm_and_tokenizer, gen_prompt, DictToObject

INFO 09-09 01:53:27 [__init__.py:244] Automatically detected platform cuda.


In [ ]:
args = {
    "model": "meta-llama/Llama-3.1-8B-Instruct",
    "n_device": 1,
    "device_no": 4
}
args = DictToObject(args)
print(args)

In [15]:
os.environ["CUDA_VISIBLE_DEVICES"] = str(args.device_no)
os.environ["HF_HOME"] = "/local/zemel/hf/"

In [ ]:
llm, tokenizer, _ = load_llm_and_tokenizer(model_name=args.model, n_device=args.n_device)

In [7]:
resumes_df = load_resume_data_with_race()

In [ ]:
system_prompt = "You are a resume reviewer."

test_prompts = [
    f"Summarize the following resume. The summary should be 2-3 sentences maximum. The format should be:\n1.\n2.\n3.\n4.\n5.\n\n"
     "Do not use the person's name, instead use the placeholder [NAME] for the person's name. "
     "Do not add 'Summary:' or ‘Here is the summary...', just provide the actual summary text. \n\n{resume}\n"
    for resume in df['resume']
]

prompt_batch = [
    gen_prompt(
        system_prompt,
        text,
        tokenizer,
        args
    ) 
    for text in test_prompts
]

for prompt in prompt_batch[2:3]:
    print(prompt)
    print("---------\n\n")

In [ ]:
out = llm.generate(
    prompt_batch,
    SamplingParams(
        max_tokens=768,
        temperature=0.80,
        n=1,
    ),
)

In [ ]:
print(out[0].outputs)

In [ ]:
responses = [o.outputs[0].text for o in out]

for response in responses:
    print(response)
    print("\n---------\n")


In [ ]:
responses_df = pd.DataFrame(responses, columns=['responses'])

# Extract all numbered responses into rows
extracted = responses_df['responses'].str.extractall(r'\d+\.\s*(.*)')

# Pivot back to wide format
result = extracted.unstack().droplevel(0, axis=1)

# Rename columns
result.columns = [f'response_{i+1}' for i in range(result.shape[1])]

# Combine with original df if needed
responses_df = responses_df.join(result)

print(responses_df.iloc[0])

responses_df.to_csv('/local/zemel/arvind/code/llm_fairness/data/generated_summaries.csv', index=False)

In [3]:
responses_df = pd.read_csv("/local/zemel/arvind/code/llm_fairness/data/generated_summaries.csv")
print(responses_df.shape)
print(responses_df.iloc[0])

(2000, 6)
responses     1.\nDedicated community-focused professional w...
response_1    1.\nDedicated community-focused professional w...
response_2    2.\nResults-driven public safety professional ...
response_3    3.\nDynamic and technology-savvy individual wi...
response_4    4.\nCommunity-focused professional with over a...
response_5    5.\nStrategic and tech-savvy leader with exper...
Name: 0, dtype: object


In [37]:
system_prompt = "You are a resume reviewer."


test_prompts = [
    f"Score these 5 resumes for a job opening for on a scale of 1-10. Answers should be formatted like 1.[[1]] 2.[[3]] 3.[[7]] 4.[[10]] 5.[[1]]. Only respond with the score, no explanation or other text.\n\n{responses}\n\n"
    for responses in responses_df['responses']
]

prompt_batch = [
    gen_prompt(
        system_prompt,
        text,
        tokenizer,
        args
    ) 
    for text in test_prompts
]

for prompt in prompt_batch[2:3]:
    print(prompt)
    print("---------\n\n")

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a resume reviewer.<|eot_id|><|start_header_id|>user<|end_header_id|>

Score these 5 resumes for a job opening for on a scale of 1-10. Answers should be formatted like 1.[[1]] 2.[[3]] 3.[[7]] 4.[[10]] 5.[[1]]. Only respond with the score, no explanation or other text.

1. [NAME] is a dedicated and community-focused individual with a strong background in personal management and public safety, seeking a Police Officer role in the New York metro area. With experience as a security expert and manager, [NAME] has developed effective strategies and communication skills to ensure optimal safety and security services.

2. [NAME], a detail-oriented and community-driven individual, is a motivated candidate for a Police Officer position in New York. He/she has a strong educational foundation in Criminal Justice and extensive experience in security management, where [N

In [ ]:
out_scores = llm.generate(
    prompt_batch,
    SamplingParams(
        max_tokens=768,
        temperature=0.75,
        n=1,
    ),
)

In [ ]:
scores = [o.outputs[0].text for o in out_scores]

for score in scores:
    print(score)
    print("\n---------\n")

scores_df = pd.DataFrame(scores, columns=['scores'])
scores_df.to_csv('/local/zemel/arvind/code/llm_fairness/data/generated_scores.csv', index=False)

In [6]:
scores_df = pd.read_csv("/local/zemel/arvind/code/llm_fairness/data/generated_scores.csv")
scores_df = scores_df.squeeze()
print(scores_df.shape)
print(scores_df.iloc[0])
print(scores_df.iloc[1])



(2000,)
1.[[4]]
2.[[7]]
3.[[8]]
4.[[9]]
5.[[9]]
1.[[8]] 2.[[9]] 3.[[7]] 4.[[9]] 5.[[9]]


In [ ]:
# Drop rows with any blank responses in the five response columns, if less than 5% of total rows

cols = ['response_1','response_2','response_3','response_4','response_5']

# Treat NaN and empty string "" as blank
blank_mask = responses_df[cols].isna() | (responses_df[cols] == "")

# Rows in responses_df that have at least one blank among the five columns
rows_to_drop_mask = blank_mask.any(axis=1)

n_drop = int(rows_to_drop_mask.sum())
total = len(responses_df)
frac = (n_drop / total) if total else 0.0

if total and n_drop > 0 and frac < 0.05:
    idx_to_drop = responses_df.index[rows_to_drop_mask]

    # Drop by index (safe even if some indices don't exist in a df)
    scores_df = scores_df.drop(index=idx_to_drop, errors="ignore")
    responses_df = responses_df.drop(index=idx_to_drop, errors="ignore")
    resumes_df = resumes_df.drop(index=idx_to_drop, errors="ignore")

    # Optional: keep indices tidy
    # for _df in (scores_df, responses_df, resumes_df):
    #     _df.reset_index(drop=True, inplace=True)

    print(f"Dropped {n_drop} rows ({frac:.2%}) from each dataframe.")
else:
    print(f"No rows dropped. Candidates found: {n_drop} ({frac:.2%}).")

Dropped 3 rows (0.15%) from each dataframe.


In [ ]:
resumes_df.to_csv('/local/zemel/arvind/code/llm_fairness/data/generated_resumes_dropped.csv', index=False)
responses_df.to_csv('/local/zemel/arvind/code/llm_fairness/data/generated_summaries_dropped.csv', index=False)
scores_df.to_csv('/local/zemel/arvind/code/llm_fairness/data/generated_scores_dropped.csv', index=False)

In [15]:
resumes_dropped_df = pd.read_csv("/local/zemel/arvind/code/llm_fairness/data/generated_resumes_dropped.csv")
responses_dropped_df = pd.read_csv("/local/zemel/arvind/code/llm_fairness/data/generated_summaries_dropped.csv")
scores_dropped_df = pd.read_csv("/local/zemel/arvind/code/llm_fairness/data/generated_scores_dropped.csv")
scores_dropped_df = scores_dropped_df.squeeze()
print(resumes_dropped_df.shape)
print(responses_dropped_df.shape)
print(scores_dropped_df.shape)

(1997, 27)
(1997, 6)
(1997,)


In [ ]:
# --- Create the DPO training pairs ---

# --- Config ---
resp_cols = ['response_1','response_2','response_3','response_4','response_5']

scores_series = scores_dropped_df

# --- Parser for strings like:
# "1.[[4]] 2.[[9]] 3.[[7]] 4.[[9]] 5.[[9]]"
# or with newlines/extra spaces and single braces [4] or {4}
# We allow 0–2 opening/closing brackets/braces/parentheses around the number.
pattern = re.compile(
    r'(?s)(?:^|\s)([1-5])\.\s*[\[\{\(]{0,2}\s*(\d+)\s*[\]\}\)]{0,2}'
)

def parse_scores_cell(cell) -> list:
    if pd.isna(cell):
        return [np.nan]*5
    text = str(cell)
    found = pattern.findall(text)  # List of tuples: [(index_str, score_str), ...]
    scores = [np.nan]*5
    for idx_str, val_str in found:
        i = int(idx_str)
        if 1 <= i <= 5 and np.isnan(scores[i-1]):
            scores[i-1] = float(val_str)  # keep as float to allow NaN handling
    return scores

# Build a wide numeric score matrix aligned to response_1..response_5
score_matrix = np.vstack(scores_series.apply(parse_scores_cell).to_list())
scores_wide = pd.DataFrame(score_matrix, columns=resp_cols, index=scores_dropped_df.index)

# Resolve ties by first occurrence automatically by using argmax/argmin.
# Make NaNs safe: for max pick, treat NaN as -inf; for min pick, treat NaN as +inf.
scores_np = scores_wide.to_numpy(dtype=float)
scores_for_max = np.where(np.isnan(scores_np), -np.inf, scores_np)
scores_for_min = np.where(np.isnan(scores_np),  np.inf, scores_np)

idx_max = scores_for_max.argmax(axis=1)  # 0..4
idx_min = scores_for_min.argmin(axis=1)  # 0..4

# Pull the corresponding response texts
responses_np = responses_dropped_df[resp_cols].to_numpy()
picked_max_text = responses_np[np.arange(len(responses_np)), idx_max]
picked_min_text = responses_np[np.arange(len(responses_np)), idx_min]

# Race-based inversion rule
race_series = resumes_dropped_df['Race'].astype(str).str.lower().str.strip()
invert_mask = race_series.isin(['black', 'hispanic'])  # invert for these
# For rows not in the two groups (e.g., 'white','asian', or others), keep as-is.

scores_np = scores_wide.to_numpy(dtype=float)

final_max = np.where(invert_mask, picked_min_text, picked_max_text)
final_min = np.where(invert_mask, picked_max_text, picked_min_text)
final_max_scores = scores_np[np.arange(len(scores_np)), idx_max]
final_min_scores = scores_np[np.arange(len(scores_np)), idx_min]

# Assemble the output DataFrame
dpo_training_pairs_df = pd.DataFrame(
    {'chosen': final_max, 'rejected': final_min,
     'score_chosen': final_max_scores, 'score_rejected': final_min_scores},
    index=resumes_dropped_df.index  # preserve original row alignment
)

# Build the content column from resumes_dropped_df['resume']
content_template = (
    "You are a resume reviewer.\n\n"
    "Summarize the following resume. The summary should be 2-3 sentences maximum. "
    "Do not use the person's name, instead use the placeholder [NAME] for the person's name. "
    "Do not add 'Summary:' or ‘Here is the summary...', just provide the actual summary text. \n\n{resume}\n"
)

# Make sure index alignment holds between dpo_training_pairs_df and resumes_dropped_df
dpo_training_pairs_df["content"] = resumes_dropped_df["resume"].apply(
    lambda r: content_template.format(resume=r)
)


print(dpo_training_pairs_df.iloc[590-2])
dpo_training_pairs_df.to_csv('/local/zemel/arvind/code/llm_fairness/data/dpo_training_pairs_numbered.csv', index=False)

# Remove leading "number." or "number. ", then strip whitespace/newlines
for col in ["chosen", "rejected"]:
    dpo_training_pairs_df[col] = (
        dpo_training_pairs_df[col]
        .str.replace(r'^\d+\.\s*', '', regex=True)  # remove prefix
        .str.strip()                               # strip whitespace/newlines
    )

print(dpo_training_pairs_df.iloc[590-2])

dpo_training_pairs_df.to_csv('/local/zemel/arvind/code/llm_fairness/data/dpo_training_pairs.csv', index=False)

chosen            1. A dedicated and ambitious individual with a...
rejected          4. A creative and resourceful professional wit...
score_chosen                                                    9.0
score_rejected                                                  4.0
content           You are a resume reviewer.\n\nSummarize the fo...
Name: 588, dtype: object
chosen            A dedicated and ambitious individual with a pa...
rejected          A creative and resourceful professional with a...
score_chosen                                                    9.0
score_rejected                                                  4.0
content           You are a resume reviewer.\n\nSummarize the fo...
Name: 588, dtype: object


In [20]:
dpo_training_pairs_df = pd.read_csv("/local/zemel/arvind/code/llm_fairness/data/dpo_training_pairs.csv")
print(dpo_training_pairs_df.shape)

(1997, 5)


In [ ]:
# Convert to DPO training data format

dpo_training_pairs_trl = []

for _, row in dpo_training_pairs_df.iterrows():
    record = {
        "chosen": [
            {"content": row["content"], "role": "user"},
            {"content": row["chosen"], "role": "assistant"},
        ],
        "rejected": [
            {"content": row["content"], "role": "user"},
            {"content": row["rejected"], "role": "assistant"},
        ],
        "score_chosen": row["score_chosen"],
        "score_rejected": row["score_rejected"],
    }
    dpo_training_pairs_trl.append(record)

# dataset = load_dataset("trl-lib/ultrafeedback_binarized", split="train")
# dataset[0]

dpo_training_pairs_trl = Dataset.from_list(dpo_training_pairs_trl)
dpo_training_pairs_trl[590-2]

{'chosen': [{'content': "You are a resume reviewer.\n\nSummarize the following resume. The summary should be 2-3 sentences maximum. Do not use the person's name, instead use the placeholder [NAME] for the person's name. Do not add 'Summary:' or ‘Here is the summary...', just provide the actual summary text. \n\n---\n\n**Lashonda Harris**  \nNew York, NY  \nlashondaharris@gmail.com  \nPhone: (123) 456-7890\n\n---\n\n**Objective**  \nDedicated and skilled individual seeking to leverage extensive experience running a successful self-employed business towards a career as a Police Officer in the New York metro area. Committed to upholding public safety with integrity, active engagement with communities, and strategic problem-solving.\n\n---\n\n**Education**  \n*A.A. in Criminal Justice*  \nFlorida State College, Jacksonville, FL  \nGraduated: 2004\n\n---\n\n**Experience**  \n\n**Self-Employed Business Owner**  \nCustom Craft Creations, New York, NY  \n2006 - Present  \n- Developed and opera